# PART 4. 데이터 병합 (merge)

## 수업 목표
- 분리된 9개 CSV 파일을 하나의 분석용 통합 데이터로 합칩니다.
- merge가 무엇인지, 왜 행이 늘어나는지 이해합니다.
- 코드를 실행하면서 컬럼이 누적되는 흐름을 눈으로 확인합니다.

## 수업 진행 포인트
| 구분 | 설명 |
|---|---|
| 데이터 관점 | ID 컬럼을 기준으로 표를 이어붙이는 것이 merge입니다. |
| 코드 관점 | pd.merge() 사용법과 how="left" 의미를 이해합니다. |
| AI 관점 | 통합 데이터가 있어야 다양한 AI 모델 학습이 가능합니다. |
| 강사 메모 | 주문 1건에 상품 여러 개이면 행이 늘어난다는 점을 반드시 설명하세요. |

---

## 병합 순서
```
orders → customers → order_items → products → category → payments → reviews → sellers
```

> 매 병합 단계마다 **크기(shape), 새로 추가된 컬럼, 누적 컬럼 목록**을 확인합니다.

In [1]:
# 셀 1. Google Drive 연결
# Colab은 런타임이 초기화되면 파일이 사라질 수 있으므로 Drive를 연결합니다.

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 셀 2. 라이브러리 및 경로 설정
# 모든 PART에서 같은 폴더를 사용해야 파일 경로 오류가 줄어듭니다.

import os
import pandas as pd

base_path = "/content/drive/MyDrive/Olist/data"
os.makedirs(base_path, exist_ok=True)

print("사용할 데이터 폴더:", base_path)

사용할 데이터 폴더: /content/drive/MyDrive/Olist/data


In [3]:
# 셀 3. 원본 CSV 파일 불러오기

customers   = pd.read_csv(f"{base_path}/olist_customers_dataset.csv")
orders      = pd.read_csv(f"{base_path}/olist_orders_dataset.csv")
order_items = pd.read_csv(f"{base_path}/olist_order_items_dataset.csv")
products    = pd.read_csv(f"{base_path}/olist_products_dataset.csv")
payments    = pd.read_csv(f"{base_path}/olist_order_payments_dataset.csv")
reviews     = pd.read_csv(f"{base_path}/olist_order_reviews_dataset.csv")
sellers     = pd.read_csv(f"{base_path}/olist_sellers_dataset.csv")
category    = pd.read_csv(f"{base_path}/product_category_name_translation.csv")

print("원본 데이터 불러오기 완료!")
print()

# 병합 전 각 테이블 크기 확인
print(f"{'테이블':<15} {'행':>8}  {'열':>4}  설명")
print("-" * 55)
for name, df_, desc in [
    ("orders",      orders,      "주문 정보"),
    ("customers",   customers,   "고객 정보"),
    ("order_items", order_items, "주문 상품 정보  ← 상품이 여러 개면 행이 많음"),
    ("products",    products,    "상품 정보"),
    ("payments",    payments,    "결제 정보"),
    ("reviews",     reviews,     "리뷰 정보"),
    ("sellers",     sellers,     "판매자 정보"),
    ("category",    category,    "카테고리 번역"),
]:
    print(f"  {name:<13}: {df_.shape[0]:>8,}행  {df_.shape[1]:>3}열  {desc}")

원본 데이터 불러오기 완료!

테이블                    행     열  설명
-------------------------------------------------------
  orders       :   99,441행    8열  주문 정보
  customers    :   99,441행    5열  고객 정보
  order_items  :  112,650행    7열  주문 상품 정보  ← 상품이 여러 개면 행이 많음
  products     :   32,951행    9열  상품 정보
  payments     :  103,886행    5열  결제 정보
  reviews      :   99,224행    7열  리뷰 정보
  sellers      :    3,095행    4열  판매자 정보
  category     :       71행    2열  카테고리 번역


| 중요도 | 테이블 | 이유 |
|---|---|---|
| ⭐⭐⭐ 매우 중요 | orders | 모든 분석의 기준이 되는 주문 테이블 |
| ⭐⭐⭐ 매우 중요 | order_items | 상품별 가격·판매자 정보 |
| ⭐⭐ 중요 | customers | 고객 위치 분석 |
| ⭐⭐ 중요 | products | 상품 카테고리·무게 |
| ⭐⭐ 중요 | payments | 결제 수단·금액 |
| ⭐⭐ 중요 | reviews | 리뷰 점수·텍스트 (AI 감성분석 활용) |
| ⭐ 보조 | sellers | 판매자 위치 |
| ⭐ 보조 | category | 카테고리 한글→영어 번역 |

---
## 누적 컬럼 확인 함수 정의

> 매 병합 후 이 함수를 호출하면 **행/열 크기 변화**와 **누적 컬럼 목록**을 한눈에 볼 수 있습니다.

In [4]:
# 셀 4. 누적 컬럼 시각화 헬퍼 함수
# 매 STEP마다 이 함수를 호출해서 컬럼이 쌓이는 과정을 확인합니다.

def show_merge_result(df, step, new_cols, before_rows=None):
    """
    병합 결과를 시각적으로 출력하는 함수
    - step     : 현재 단계 이름 (예: "STEP 1. orders + customers")
    - new_cols : 이번 병합에서 새로 추가된 컬럼 목록
    - before_rows : 병합 전 행 수 (행이 늘어났는지 확인용)
    """
    total_rows = len(df)
    total_cols = df.shape[1]

    print(f"{'='*55}")
    print(f"  {step}")
    print(f"{'='*55}")

    # 행 변화
    if before_rows is not None:
        diff = total_rows - before_rows
        arrow = f"(+{diff:,} 증가 ← 1:N 관계!)" if diff > 0 else "(변화 없음)"
        print(f"  행(Row) : {before_rows:>10,}  →  {total_rows:>10,}  {arrow}")
    else:
        print(f"  행(Row) : {total_rows:>10,}")

    print(f"  열(Col) : {total_cols:>10}")
    print()

    # 새로 추가된 컬럼
    print(f"  ✅ 새로 추가된 컬럼 ({len(new_cols)}개):")
    for c in new_cols:
        print(f"       + {c}")
    print()

    # 누적 컬럼 전체 목록
    print(f"  📋 누적 컬럼 전체 목록 (총 {total_cols}개):")
    for i, col in enumerate(df.columns, 1):
        mark = " ← NEW" if col in new_cols else ""
        print(f"    {i:>3}. {col}{mark}")
    print(f"{'='*55}")
    print()

print("헬퍼 함수 정의 완료! 이제 병합을 시작합니다.")

헬퍼 함수 정의 완료! 이제 병합을 시작합니다.


---
## STEP 1. orders + customers 병합
- 기준 컬럼: `customer_id`
- 추가되는 컬럼: `customer_unique_id`, `customer_zip_code_prefix`, `customer_city`, `customer_state`

> orders가 99,441행이므로 병합 후에도 **행 수는 그대로** 유지됩니다.  
> 고객 1명 = 주문 1건 기준이기 때문입니다. (1:1 관계)

In [5]:
# 셀 5. STEP 1 - orders + customers 병합
# on="customer_id" : 두 테이블에 공통으로 있는 컬럼을 기준으로 연결
# how="left"       : orders를 기준으로, 매칭이 안 되어도 orders 행은 유지

df = pd.merge(orders, customers, on="customer_id", how="left")

new_cols_step1 = [c for c in customers.columns if c != "customer_id"]
show_merge_result(df, "STEP 1. orders + customers", new_cols_step1)

df.head(3)

  STEP 1. orders + customers
  행(Row) :     99,441
  열(Col) :         12

  ✅ 새로 추가된 컬럼 (4개):
       + customer_unique_id
       + customer_zip_code_prefix
       + customer_city
       + customer_state

  📋 누적 컬럼 전체 목록 (총 12개):
      1. order_id
      2. customer_id
      3. order_status
      4. order_purchase_timestamp
      5. order_approved_at
      6. order_delivered_carrier_date
      7. order_delivered_customer_date
      8. order_estimated_delivery_date
      9. customer_unique_id ← NEW
     10. customer_zip_code_prefix ← NEW
     11. customer_city ← NEW
     12. customer_state ← NEW



,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO


---
## STEP 2. order_items 병합
- 기준 컬럼: `order_id`
- 추가되는 컬럼: `order_item_id`, `product_id`, `seller_id`, `shipping_limit_date`, `price`, `freight_value`

> ⚠️ **행이 늘어납니다!** 주문 1개에 상품이 여러 개면 같은 order_id로 여러 행이 생깁니다.  
> 예) 치킨 + 콜라 주문 → 같은 order_id로 **2행** 생성 (1:N 관계)

In [6]:
# 셀 6. STEP 2 - order_items 병합

before = len(df)
df = pd.merge(df, order_items, on="order_id", how="left")

new_cols_step2 = [c for c in order_items.columns if c != "order_id"]
show_merge_result(df, "STEP 2. + order_items", new_cols_step2, before_rows=before)

df.head(3)

  STEP 2. + order_items
  행(Row) :     99,441  →     113,425  (+13,984 증가 ← 1:N 관계!)
  열(Col) :         18

  ✅ 새로 추가된 컬럼 (6개):
       + order_item_id
       + product_id
       + seller_id
       + shipping_limit_date
       + price
       + freight_value

  📋 누적 컬럼 전체 목록 (총 18개):
      1. order_id
      2. customer_id
      3. order_status
      4. order_purchase_timestamp
      5. order_approved_at
      6. order_delivered_carrier_date
      7. order_delivered_customer_date
      8. order_estimated_delivery_date
      9. customer_unique_id
     10. customer_zip_code_prefix
     11. customer_city
     12. customer_state
     13. order_item_id ← NEW
     14. product_id ← NEW
     15. seller_id ← NEW
     16. shipping_limit_date ← NEW
     17. price ← NEW
     18. freight_value ← NEW



,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22


---
## STEP 3. products 병합
- 기준 컬럼: `product_id`
- 추가되는 컬럼: 상품 카테고리명, 상품명 길이, 무게, 크기 등

> 상품 1개 = 1행이므로 행 수는 그대로입니다. (1:1 관계)

In [7]:
# 셀 7. STEP 3 - products 병합

before = len(df)
df = pd.merge(df, products, on="product_id", how="left")

new_cols_step3 = [c for c in products.columns if c != "product_id"]
show_merge_result(df, "STEP 3. + products", new_cols_step3, before_rows=before)

df[["order_id", "product_id", "product_category_name", "product_weight_g", "price"]].head(3)

  STEP 3. + products
  행(Row) :    113,425  →     113,425  (변화 없음)
  열(Col) :         26

  ✅ 새로 추가된 컬럼 (8개):
       + product_category_name
       + product_name_lenght
       + product_description_lenght
       + product_photos_qty
       + product_weight_g
       + product_length_cm
       + product_height_cm
       + product_width_cm

  📋 누적 컬럼 전체 목록 (총 26개):
      1. order_id
      2. customer_id
      3. order_status
      4. order_purchase_timestamp
      5. order_approved_at
      6. order_delivered_carrier_date
      7. order_delivered_customer_date
      8. order_estimated_delivery_date
      9. customer_unique_id
     10. customer_zip_code_prefix
     11. customer_city
     12. customer_state
     13. order_item_id
     14. product_id
     15. seller_id
     16. shipping_limit_date
     17. price
     18. freight_value
     19. product_category_name ← NEW
     20. product_name_lenght ← NEW
     21. product_description_lenght ← NEW
     22. product_photos_qty ← NEW
     23. p

,order_id,product_id,product_category_name,product_weight_g,price
0,e481f51cbdc54678b7cc49136f2d6af7,87285b34884572647811a353c7ac498a,utilidades_domesticas,500.0,29.99
1,53cdb2fc8bc7dce0b6741e2150273451,595fac2a385ac33a80bd5114aec74eb8,perfumaria,400.0,118.70
2,47770eb9100c2d0c44946d9cf07ec65d,aa4383b373c6aca5d8797843e5594415,automotivo,420.0,159.90


---
## STEP 4. category 병합 (카테고리 영문명 추가)
- 기준 컬럼: `product_category_name`
- 추가되는 컬럼: `product_category_name_english`
- 브라질 포르투갈어 카테고리명 → 영어 카테고리명으로 변환

In [8]:
# 셀 8. STEP 4 - category 병합

before = len(df)
df = pd.merge(df, category, on="product_category_name", how="left")

new_cols_step4 = ["product_category_name_english"]
show_merge_result(df, "STEP 4. + category (영문 카테고리)", new_cols_step4, before_rows=before)

# 포르투갈어 → 영어 변환 확인
df[["product_category_name", "product_category_name_english"]].dropna().drop_duplicates().head(5)

  STEP 4. + category (영문 카테고리)
  행(Row) :    113,425  →     113,425  (변화 없음)
  열(Col) :         27

  ✅ 새로 추가된 컬럼 (1개):
       + product_category_name_english

  📋 누적 컬럼 전체 목록 (총 27개):
      1. order_id
      2. customer_id
      3. order_status
      4. order_purchase_timestamp
      5. order_approved_at
      6. order_delivered_carrier_date
      7. order_delivered_customer_date
      8. order_estimated_delivery_date
      9. customer_unique_id
     10. customer_zip_code_prefix
     11. customer_city
     12. customer_state
     13. order_item_id
     14. product_id
     15. seller_id
     16. shipping_limit_date
     17. price
     18. freight_value
     19. product_category_name
     20. product_name_lenght
     21. product_description_lenght
     22. product_photos_qty
     23. product_weight_g
     24. product_length_cm
     25. product_height_cm
     26. product_width_cm
     27. product_category_name_english ← NEW



,product_category_name,product_category_name_english
0,utilidades_domesticas,housewares
1,perfumaria,perfumery
2,automotivo,auto
3,pet_shop,pet_shop
4,papelaria,stationery


---
## STEP 5. payments 병합
- 기준 컬럼: `order_id`
- 추가되는 컬럼: `payment_sequential`, `payment_type`, `payment_installments`, `payment_value`

> ⚠️ **행이 또 늘어날 수 있습니다!**  
> 한 주문에 신용카드 + 바우처처럼 결제 수단이 2개 이상이면 행이 추가됩니다. (1:N 관계)

In [9]:
# 셀 9. STEP 5 - payments 병합

before = len(df)
df = pd.merge(df, payments, on="order_id", how="left")

new_cols_step5 = [c for c in payments.columns if c != "order_id"]
show_merge_result(df, "STEP 5. + payments", new_cols_step5, before_rows=before)

df[["order_id", "payment_type", "payment_installments", "payment_value"]].head(3)

  STEP 5. + payments
  행(Row) :    113,425  →     118,434  (+5,009 증가 ← 1:N 관계!)
  열(Col) :         31

  ✅ 새로 추가된 컬럼 (4개):
       + payment_sequential
       + payment_type
       + payment_installments
       + payment_value

  📋 누적 컬럼 전체 목록 (총 31개):
      1. order_id
      2. customer_id
      3. order_status
      4. order_purchase_timestamp
      5. order_approved_at
      6. order_delivered_carrier_date
      7. order_delivered_customer_date
      8. order_estimated_delivery_date
      9. customer_unique_id
     10. customer_zip_code_prefix
     11. customer_city
     12. customer_state
     13. order_item_id
     14. product_id
     15. seller_id
     16. shipping_limit_date
     17. price
     18. freight_value
     19. product_category_name
     20. product_name_lenght
     21. product_description_lenght
     22. product_photos_qty
     23. product_weight_g
     24. product_length_cm
     25. product_height_cm
     26. product_width_cm
     27. product_category_name_english
  

,order_id,payment_type,payment_installments,payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,credit_card,1.0,18.12
1,e481f51cbdc54678b7cc49136f2d6af7,voucher,1.0,2.00
2,e481f51cbdc54678b7cc49136f2d6af7,voucher,1.0,18.59


---
## STEP 6. reviews 병합
- 기준 컬럼: `order_id`
- 추가되는 컬럼: `review_id`, `review_score`, `review_comment_title`, `review_comment_message` 등

> `review_comment_message`는 텍스트 데이터로, **AI 감성 분석 모델**의 학습 재료가 됩니다.

In [10]:
# 셀 10. STEP 6 - reviews 병합

before = len(df)
df = pd.merge(df, reviews, on="order_id", how="left")

new_cols_step6 = [c for c in reviews.columns if c != "order_id"]
show_merge_result(df, "STEP 6. + reviews", new_cols_step6, before_rows=before)

df[["order_id", "review_score", "review_comment_message"]].dropna(subset=["review_comment_message"]).head(3)

  STEP 6. + reviews
  행(Row) :    118,434  →     119,143  (+709 증가 ← 1:N 관계!)
  열(Col) :         37

  ✅ 새로 추가된 컬럼 (6개):
       + review_id
       + review_score
       + review_comment_title
       + review_comment_message
       + review_creation_date
       + review_answer_timestamp

  📋 누적 컬럼 전체 목록 (총 37개):
      1. order_id
      2. customer_id
      3. order_status
      4. order_purchase_timestamp
      5. order_approved_at
      6. order_delivered_carrier_date
      7. order_delivered_customer_date
      8. order_estimated_delivery_date
      9. customer_unique_id
     10. customer_zip_code_prefix
     11. customer_city
     12. customer_state
     13. order_item_id
     14. product_id
     15. seller_id
     16. shipping_limit_date
     17. price
     18. freight_value
     19. product_category_name
     20. product_name_lenght
     21. product_description_lenght
     22. product_photos_qty
     23. product_weight_g
     24. product_length_cm
     25. product_height_cm
     26

,order_id,review_score,review_comment_message
0,e481f51cbdc54678b7cc49136f2d6af7,4.0,"Não testei o produto ainda, mas ele veio corre..."
1,e481f51cbdc54678b7cc49136f2d6af7,4.0,"Não testei o produto ainda, mas ele veio corre..."
2,e481f51cbdc54678b7cc49136f2d6af7,4.0,"Não testei o produto ainda, mas ele veio corre..."


---
## STEP 7. sellers 병합
- 기준 컬럼: `seller_id`
- 추가되는 컬럼: `seller_zip_code_prefix`, `seller_city`, `seller_state`

In [11]:
# 셀 11. STEP 7 - sellers 병합 (마지막 단계)

before = len(df)
df = pd.merge(df, sellers, on="seller_id", how="left")

new_cols_step7 = [c for c in sellers.columns if c != "seller_id"]
show_merge_result(df, "STEP 7. + sellers  ★ 최종 병합 완료!", new_cols_step7, before_rows=before)

df[["order_id", "seller_id", "seller_city", "seller_state"]].head(3)

  STEP 7. + sellers  ★ 최종 병합 완료!
  행(Row) :    119,143  →     119,143  (변화 없음)
  열(Col) :         40

  ✅ 새로 추가된 컬럼 (3개):
       + seller_zip_code_prefix
       + seller_city
       + seller_state

  📋 누적 컬럼 전체 목록 (총 40개):
      1. order_id
      2. customer_id
      3. order_status
      4. order_purchase_timestamp
      5. order_approved_at
      6. order_delivered_carrier_date
      7. order_delivered_customer_date
      8. order_estimated_delivery_date
      9. customer_unique_id
     10. customer_zip_code_prefix
     11. customer_city
     12. customer_state
     13. order_item_id
     14. product_id
     15. seller_id
     16. shipping_limit_date
     17. price
     18. freight_value
     19. product_category_name
     20. product_name_lenght
     21. product_description_lenght
     22. product_photos_qty
     23. product_weight_g
     24. product_length_cm
     25. product_height_cm
     26. product_width_cm
     27. product_category_name_english
     28. payment_sequential
    

,order_id,seller_id,seller_city,seller_state
0,e481f51cbdc54678b7cc49136f2d6af7,3504c0cb71d7fa48d967e0e4c94d59d9,maua,SP
1,e481f51cbdc54678b7cc49136f2d6af7,3504c0cb71d7fa48d967e0e4c94d59d9,maua,SP
2,e481f51cbdc54678b7cc49136f2d6af7,3504c0cb71d7fa48d967e0e4c94d59d9,maua,SP


---
## 병합 완료 — 최종 확인

In [12]:
# 셀 12. 최종 병합 결과 요약

print("=" * 50)
print("           최종 통합 데이터 요약")
print("=" * 50)
print(f"  전체 행(데이터) 수 : {len(df):>10,}")
print(f"  고유 주문 수       : {df['order_id'].nunique():>10,}")
print(f"  고유 고객 수       : {df['customer_unique_id'].nunique():>10,}")
print(f"  고유 상품 수       : {df['product_id'].nunique():>10,}")
print(f"  전체 컬럼 수       : {df.shape[1]:>10}")
print("=" * 50)
print()
print("💡 행 수(약 119,143) > 주문 수(99,441)인 이유:")
print("   주문 1건에 상품 또는 결제 정보가 여러 개이면")
print("   행이 늘어나기 때문입니다. 오류가 아닌 정상 결과입니다.")

           최종 통합 데이터 요약
  전체 행(데이터) 수 :    119,143
  고유 주문 수       :     99,441
  고유 고객 수       :     96,096
  고유 상품 수       :     32,951
  전체 컬럼 수       :         40

💡 행 수(약 119,143) > 주문 수(99,441)인 이유:
   주문 1건에 상품 또는 결제 정보가 여러 개이면
   행이 늘어나기 때문입니다. 오류가 아닌 정상 결과입니다.


In [13]:
# 셀 13. 통합 데이터 저장
# 다음 PART에서 매번 병합하지 않도록 저장합니다.
# index=False : 행 번호(0,1,2...)는 저장하지 않습니다.

save_path = f"{base_path}/olist_master_data.csv"
df.to_csv(save_path, index=False)

print("저장 완료!")
print("저장 위치:", save_path)
print(f"파일 크기: {os.path.getsize(save_path)/1024/1024:.1f} MB")

저장 완료!
저장 위치: /content/drive/MyDrive/Olist/data/olist_master_data.csv
파일 크기: 62.7 MB


---

## 마무리 정리

| 확인할 내용 | 설명 |
|---|---|
| 이 파트의 핵심 | 9개 CSV를 1개 통합 파일(olist_master_data.csv)로 합칩니다. |
| 핵심 함수 | `pd.merge(왼쪽, 오른쪽, on=기준컬럼, how="left")` |
| 행이 늘어나는 이유 | 1:N 관계에서 N쪽 데이터가 여러 행으로 펼쳐지기 때문 |
| 결과를 이해하는 법 | 숫자만 보지 말고, 어떤 컬럼이 새로 붙었는지 확인하세요. |
| AI 연결 | 데이터 구조를 이해해야 모델 학습 결과도 해석할 수 있습니다. |
| 다음 단계 | PART 5에서 통합 데이터로 EDA 분석을 진행합니다. |